# 结合关键词和向量检索

关键词检索擅长命中原文术语，向量检索擅长理解不同说法。本实验比较关键词、向量和两者合并三种结果。


## 怎样比较

第一个问题同时询问“模型评估与选择”以及类别不平衡时的宏平均和微平均。必要资料分别在第 18 页和第 21 页；如果关键词检索和向量检索各自找回一页，合并才算真正补全了资料。三种方法使用同一个问题、同一份片段库、每路前 20 个候选，最后都只保留 2 个片段。

代码直接使用教程随附的 BGE 向量库，只计算新问题的向量，不会重新计算 987 个文档片段的向量。如果运行环境缺少所需的软件，会明确报错，不会悄悄换成另一种向量算法。

这里检查必要资料页是否被找到，不把检索到的两段文字当成完整回答。每种方法最多给回答模型 512 个字符；另一个核函数问题用于检查两种检索都已找到正确资料时，合并是否还有变化。


In [1]:
import json
import sys
from pathlib import Path

def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到课程根目录")

course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import (
    build_bm25_chunk_search,
    build_default_chunk_search,
    load_query_catalog,
    load_default_chunks,
    load_default_collection,
)
from common.nontraining_utils import load_annotation

collection = load_default_collection()
chunks = load_default_chunks(collection)
keyword_search = build_bm25_chunk_search(chunks)
cases = {case["id"]: case for case in load_query_catalog()}
main_case = cases["model_evaluation_and_macro_micro"]
boundary_case = cases["svm_kernel_evidence"]

candidate_k = 20
return_k = 2
chunk_limit = max(len(chunk["text"]) for chunk in chunks)
context_budget = return_k * chunk_limit

vector_search = build_default_chunk_search(collection)
vector_backend = "教程随附的 BGE 向量库"

print(f"资料库：{len(chunks)} 个片段，{collection.count()} 个向量条目")
print(f"向量实现：{vector_backend}")
print(
    f"候选范围：关键词和向量各取前 {candidate_k} 条；混合方法会执行两次检索"
)
print(
    f"最终输出限制：返回 {return_k} 条，单片段上限 {chunk_limit} 字，"
    f"回答上下文上限 {context_budget} 字"
)


资料库：987 个片段，987 个向量条目
向量实现：教程随附的 BGE 向量库
候选范围：关键词和向量各取前 20 条；混合方法会执行两次检索
最终输出限制：返回 2 条，单片段上限 256 字，回答上下文上限 512 字


## 主问题：两页必要资料能否互相补足

三种方法使用同一个问题和同一份资料。关键词检索和向量检索各取前 20 条，混合方法运行这两路检索后再合并结果，因此检索成本更高；三种方法最终都只把 2 个片段交给回答。合并时，第 r 名得到 1/(60+r) 分；60 只是固定的平滑值，用来缩小相邻名次的分差，本实验没有针对问题调这个数。这里比较目标页覆盖和实际字符数。


In [2]:
from common.eval_utils import emit_tutorial_audit

def merge_by_rank(rankings, keep=2, offset=60):
    scores = {}
    items = {}
    for ranking in rankings:
        for rank, item in enumerate(ranking, start=1):
            scores[item.chunk_id] = scores.get(item.chunk_id, 0.0)
            scores[item.chunk_id] += 1.0 / (offset + rank)
            items[item.chunk_id] = item
    ordered = sorted(
        items,
        key=lambda chunk_id: (-scores[chunk_id], chunk_id),
    )
    return [items[chunk_id] for chunk_id in ordered[:keep]]

def matched_pages(results, expected_pages):
    expected = set(int(page) for page in expected_pages)
    return sorted(
        expected
        & {page for item in results for page in item.pages}
    )

def context_chars(results):
    return sum(len(item.text) for item in results)

def result_metrics(results, expected_pages):
    pages = [int(page) for item in results for page in item.pages]
    expected = set(int(page) for page in expected_pages)
    found = expected.intersection(pages)
    return {
        'pages': pages,
        'first_required_rank': next((index for index, item in enumerate(results, 1) if expected.intersection(item.pages)), None),
        'required_page_coverage': len(found) / len(expected) if expected else 0.0,
    }

def summarize_results(label, results, expected_pages):
    hit_pages = matched_pages(results, expected_pages)
    chars = context_chars(results)
    print(
        f"{label}：返回页码 "
        f"{[page for item in results for page in item.pages]}；"
        f"必要资料页命中 {len(hit_pages)}/{len(expected_pages)} "
        f"{hit_pages}；上下文字符数 {chars}"
    )
    assert len(results) == return_k
    assert chars <= context_budget
    return len(hit_pages), chars

main_query = main_case["query"]
main_keyword_candidates = keyword_search(main_query, top_k=candidate_k)
main_vector_candidates = vector_search(main_query, top_k=candidate_k)
main_hybrid_results = merge_by_rank(
    [main_keyword_candidates, main_vector_candidates],
    keep=return_k,
)
main_keyword_results = main_keyword_candidates[:return_k]
main_vector_results = main_vector_candidates[:return_k]
main_annotation = load_annotation(main_case["id"])
main_expected_pages = main_annotation["expected_pages"]

print(f"用户问题：{main_query}")
print(f"预期必要资料页：{main_expected_pages}")
print(
    f"关键词、向量各处理 {candidate_k} 条候选；"
    f"合并处理两份 {candidate_k} 条名单；三种方法最终均返回 {return_k} 条"
)
main_coverage = {}
main_cost = {}
for label, results in (
    ("关键词检索", main_keyword_results),
    ("向量检索", main_vector_results),
    ("按名次合并", main_hybrid_results),
):
    main_coverage[label], main_cost[label] = summarize_results(
        label, results, main_expected_pages
    )

assert main_coverage["按名次合并"] == len(main_expected_pages)
assert main_coverage["按名次合并"] > max(
    main_coverage["关键词检索"], main_coverage["向量检索"]
)
print(
    "合并首条是否复制关键词首条："
    f"{main_hybrid_results[0].chunk_id == main_keyword_results[0].chunk_id}"
)
print(
    "结论：合并后多找到至少一页必要资料；三种方法最终返回数量与上下文上限相同，"
    "但合并多执行一次检索。"
)
emit_tutorial_audit({
    'case_id': 'model_evaluation_and_macro_micro',
    'method': '结合关键词和向量检索',
    'role': 'main',
    'before': result_metrics(main_keyword_results, main_expected_pages),
    'after': result_metrics(main_hybrid_results, main_expected_pages),
})


用户问题：“模型评估与选择”是什么？为什么类别不平衡时要区分宏平均和微平均？
预期必要资料页：[18, 21]
关键词、向量各处理 20 条候选；合并处理两份 20 条名单；三种方法最终均返回 2 条
关键词检索：返回页码 [18, 99]；必要资料页命中 1/2 [18]；上下文字符数 512
向量检索：返回页码 [21, 16]；必要资料页命中 1/2 [21]；上下文字符数 512
按名次合并：返回页码 [21, 18]；必要资料页命中 2/2 [18, 21]；上下文字符数 512
合并首条是否复制关键词首条：False
结论：合并后多找到至少一页必要资料；三种方法最终返回数量与上下文上限相同，但合并多执行一次检索。



## 准确术语已经写在问题里时，先试关键词检索

Slater 是原文中的准确术语。使用完整问题检索时，关键词检索把目标资料从向量结果的第 2 条提到第 1 条。再换一道支持向量机问题检查时，两种方法都已经把正确资料排在第一条，因此没有必要声称关键词仍有额外改善。两道题都使用相同问题、相同片段库和前 20 条候选。


In [3]:
from common.eval_utils import emit_tutorial_audit

keyword_main_case = cases["slater_strong_duality"]
keyword_check_case = cases["svm_margin_generalization"]

def first_expected_rank(results, expected_pages):
    expected = set(expected_pages)
    return next(
        (rank for rank, item in enumerate(results, 1) if expected.intersection(item.pages)),
        None,
    )

keyword_comparisons = {}
keyword_results = {}
for label, current_case in (("主要问题", keyword_main_case), ("换题检查", keyword_check_case)):
    current_query = current_case["query"]
    current_vector = vector_search(current_query, top_k=candidate_k)
    current_keyword = keyword_search(current_query, top_k=candidate_k)
    current_annotation = load_annotation(current_case["id"])
    vector_rank = first_expected_rank(current_vector, current_annotation["expected_pages"])
    keyword_rank = first_expected_rank(current_keyword, current_annotation["expected_pages"])
    keyword_comparisons[label] = (vector_rank, keyword_rank)
    keyword_results[current_case['id']] = (current_vector, current_keyword)
    print(f"{label}：{current_query}")
    print(f"正确资料排名（向量 → 关键词）：{vector_rank} → {keyword_rank}")
    print(f"候选数量：{len(current_vector)} → {len(current_keyword)}")

main_vector_rank, main_keyword_rank = keyword_comparisons["主要问题"]
check_vector_rank, check_keyword_rank = keyword_comparisons["换题检查"]
assert main_keyword_rank == 1 and main_vector_rank > main_keyword_rank
assert check_vector_rank == check_keyword_rank == 1
print("结论：准确术语让主要问题更早找到正确资料；已经排在第一条的问题没有变化。")
keyword_main_vector, keyword_main_after = keyword_results['slater_strong_duality']
emit_tutorial_audit({
    'case_id': 'slater_strong_duality',
    'method': '关键词检索',
    'role': 'main',
    'before': result_metrics(keyword_main_vector, load_annotation(keyword_main_case['id'])['expected_pages']),
    'after': result_metrics(keyword_main_after, load_annotation(keyword_main_case['id'])['expected_pages']),
})
keyword_check_vector, keyword_check_after = keyword_results['svm_margin_generalization']
emit_tutorial_audit({
    'case_id': 'svm_margin_generalization',
    'method': '关键词检索',
    'role': 'check',
    'before': result_metrics(keyword_check_vector, load_annotation(keyword_check_case['id'])['expected_pages']),
    'after': result_metrics(keyword_check_after, load_annotation(keyword_check_case['id'])['expected_pages']),
    'check_purpose': '说明不适用或限制',
})


主要问题：Slater 条件如何保证强对偶成立？
正确资料排名（向量 → 关键词）：2 → 1
候选数量：20 → 20
换题检查：支持向量机为什么选择离正负样本尽可能远且位于正中间的超平面？
正确资料排名（向量 → 关键词）：1 → 1
候选数量：20 → 20
结论：准确术语让主要问题更早找到正确资料；已经排在第一条的问题没有变化。




## 正确资料已经排在前面时，合并不会额外改善

下面仍然让每种方法查看前 20 条候选、最终返回 2 条，并使用相同的文字预算。在“ SVM 为什么能用核函数处理原始空间线性不可分的问题？”上，关键词和向量检索都已把第 66 页排在前面；合并不会凭空增加新的必要资料，因此这里只把它作为限制复查。


In [4]:
from common.eval_utils import emit_tutorial_audit

boundary_query = boundary_case["query"]
boundary_keyword_candidates = keyword_search(
    boundary_query, top_k=candidate_k
)
boundary_vector_candidates = vector_search(
    boundary_query, top_k=candidate_k
)
boundary_keyword_results = boundary_keyword_candidates[:return_k]
boundary_vector_results = boundary_vector_candidates[:return_k]
boundary_hybrid_results = merge_by_rank(
    [
        boundary_keyword_candidates,
        boundary_vector_candidates,
    ],
    keep=return_k,
)
boundary_annotation = load_annotation(boundary_case["id"])
boundary_expected_pages = boundary_annotation["expected_pages"]

print(f"用户问题：{boundary_query}")
print(f"预期必要资料页：{boundary_expected_pages}")
boundary_keyword_rank = first_expected_rank(
    boundary_keyword_candidates, boundary_expected_pages
)
boundary_vector_rank = first_expected_rank(
    boundary_vector_candidates, boundary_expected_pages
)
print(
    "正确资料首次出现位置（关键词 → 向量）："
    f"{boundary_keyword_rank} → {boundary_vector_rank}"
)
boundary_coverage = {}
boundary_cost = {}
for label, results in (
    ("关键词检索", boundary_keyword_results),
    ("向量检索", boundary_vector_results),
    ("按名次合并", boundary_hybrid_results),
):
    boundary_coverage[label], boundary_cost[label] = summarize_results(
        label, results, boundary_expected_pages
    )

assert all(value == len(boundary_expected_pages) for value in boundary_coverage.values())
assert boundary_keyword_rank == 1
assert boundary_vector_rank == 1
print(
    "这道题的结论：两种检索都已找到第 66 页，"
    "合并后必要资料覆盖没有额外增加。"
)
emit_tutorial_audit({
    'case_id': 'svm_kernel_evidence',
    'method': '结合关键词和向量检索',
    'role': 'check',
    'before': result_metrics(boundary_keyword_results, boundary_expected_pages),
    'after': result_metrics(boundary_hybrid_results, boundary_expected_pages),
    'check_purpose': '说明不适用或限制',
})


用户问题：SVM 为什么能用核函数处理原始空间线性不可分的问题？请同时说明对偶形式中的内积和高维映射。
预期必要资料页：[66]
正确资料首次出现位置（关键词 → 向量）：1 → 1
关键词检索：返回页码 [66, 133]；必要资料页命中 1/1 [66]；上下文字符数 511
向量检索：返回页码 [66, 66]；必要资料页命中 1/1 [66]；上下文字符数 511
按名次合并：返回页码 [66, 66]；必要资料页命中 1/1 [66]；上下文字符数 511
这道题的结论：两种检索都已找到第 66 页，合并后必要资料覆盖没有额外增加。



## 结论与限制

主要问题用“是否找到必要资料页”判断效果，不把它当成最终答案质量。关键词前两条只覆盖第 18 页，BGE 向量前两条只覆盖第 21 页，按名次合并后两页同时进入前两条。三种方法最终返回数和 512 字上下文预算相同；混合方法需要同时运行关键词和 BGE 检索，成本高于单独使用其中一种。合并首条来自另一种检索方法，因此不是复制关键词结果。

本页保存的结果来自 BAAI/bge-small-zh-v1.5 和教程随附的向量库。页级命中不等于每个概念都已完整出现在片段正文中；若要评价完整回答，还应把更多上下文交给回答模型并单独检查要点。换题检查也说明：必要资料已经排在前面时，合并不会凭空创造额外提升。
